In [ ]:
from pathlib import Path

# Локально репозиторий уже открыт. В Colab/Kaggle загружаем его с GitHub.
hosted = Path("/kaggle/working").is_dir() or Path("/content").is_dir()
if hosted:
    work_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
    repo_dir = work_dir / "recsys-rl-sequential"
    if repo_dir.exists():
        !git -C {repo_dir} pull --ff-only
    else:
        !git clone https://github.com/daosistt/recsys-rl-sequential {repo_dir}
    %cd {repo_dir}
    %pip install -q -r requirements.txt
else:
    # Локально работаем из корня проекта.
    repo_dir = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
    %cd {repo_dir}


In [ ]:
from src import config as project_config

project_config.setup_paths()
from src.config import *
from src.prepare_data import load_yambda_data, train_val_test_split
from src.evaluation import load_model
from src.reward_model import clear_memory
from src.reward import prepare_track_statistics
from src.reinforce import prepare_rl_contexts, train_reinforce

print(f"Данные: {DATA_DIR}")
print(f"Результаты: {ARTIFACTS_DIR}")
print(f"Устройство: {device}")


In [ ]:
baseline_dir = find_artifact_dir("baseline_t4_1000_minfreq5")
reward_dir = find_artifact_dir("reward_catboost_stats_minfreq5")

print(f"Бейзлайн: {baseline_dir}")
print(f"Reward model: {reward_dir}")


## REINFORCE: один следующий трек

Политика выбирает трек случайно из своего распределения, исключая PAD и UNK. Награду считает зафиксированный CatBoost на эмбеддингах исходной GPT-2 и пяти признаках.

Минимизируем `-mean((reward - baseline) * log_prob) + kl_coef * KL(policy || reference)`. Baseline — награда жадного выбора исходной GPT-2 для того же контекста. Dropout отключён, градиенты политики сохраняются. Замороженная исходная GPT-2 также используется для эмбеддингов reward model.

В `RL_PARAMS` доступны награды `listening`, `repeat_penalty`, `popularity_penalty`; `penalty` задаёт штраф, `kl_coef` — силу ограничения отклонения от бейзлайна. Для каждого варианта заново выполняем загрузку моделей: все запуски начинаются с одного бейзлайна.

Контексты обучения берём из train. Validation и её seed фиксированы; лучший чекпоинт выбирается по `val_reward - kl_coef * val_kl`, включая исходную эпоху 0. Это оценка по reward model. Сравнение релевантности и поведения добавим отдельно; test здесь не используется.


In [ ]:
params = RL_PARAMS.copy()
torch.manual_seed(params["seed"])
np.random.seed(params["seed"])

# Каждый вариант награды начинаем с двух отдельных копий исходного бейзлайна.
policy, vocab, baseline_config = load_model(baseline_dir, device)
reference, _, _ = load_model(baseline_dir, device)
reference.requires_grad_(False)
with open(reward_dir / "reward_head.pkl", "rb") as f:
    reward_head = pickle.load(f)
with open(reward_dir / "config.json") as f:
    reward_config = json.load(f)
assert reward_config["backbone_run"] == baseline_dir.name
assert reward_config["split"] == baseline_config["split"]

output_dir = ARTIFACTS_DIR / f"rl_{params['reward_type']}_seed{params['seed']}"
experiment = {"baseline_run": baseline_dir.name, "reward_run": reward_dir.name,
              "baseline_config": baseline_config}
print(f"Награда: {params['reward_type']}, KL: {params['kl_coef']}")
print(f"Сохраним в: {output_dir}")


In [ ]:
df = load_yambda_data(size=baseline_config["training"]["data_size"])
train, val, test = train_val_test_split(df, **baseline_config["split"])
del df, test
clear_memory()

statistics = prepare_track_statistics(train)
train_contexts = prepare_rl_contexts(train, vocab, params["train_samples"], params["seed"])
val_contexts = prepare_rl_contexts(val, vocab, params["val_samples"], params["seed"], context=train)
del train, val
clear_memory()
print(f"Контексты: train={len(train_contexts)}, validation={len(val_contexts)}")


In [ ]:
# Эта ячейка запускает обучение. Исходный бейзлайн и reward model не перезаписываются.
history = train_reinforce(
    policy, reference, reward_head, vocab, train_contexts, val_contexts,
    statistics, params, output_dir, experiment,
)
history
